In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET]
X = train.drop(columns=[TARGET])


In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

y = train[TARGET]
X = train.drop(columns=[TARGET])

# find categorical columns
cat_features = X.select_dtypes("object").columns.tolist()

model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    eval_metric="AUC",
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=42,
    early_stopping_rounds=200
)

model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=200
)

pred_probs = model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpu.csv", index=False)

## Catboost Optimization

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"

test_ids = test["id"]

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes("object").columns.tolist()


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=8,

    eval_metric="AUC",
    loss_function="Logloss",

    task_type="GPU",
    devices="0",

    random_seed=42,

    early_stopping_rounds=200,

    verbose=200
)


model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)


pred_probs = model.predict_proba(test)[:, 1]


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_gpuV1.1.csv", index=False)

print("submission_catboost_gpuV1.1.csv created")

## Catboost with Stratifier K-fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
cat_features = X.select_dtypes(include=["object","string"]).columns.tolist()


N_FOLDS = 5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== Fold {fold+1} =====")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(

        iterations=5000,
        learning_rate=0.03,
        depth=8,

        eval_metric="AUC",
        loss_function="Logloss",

        task_type="GPU",
        devices="0",

        random_seed=42,

        early_stopping_rounds=200,

        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features
    )

    val_pred = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(test)[:,1] / N_FOLDS

y_binary = y.map({"No":0,"Yes":1})
auc = roc_auc_score(y_binary, oof_preds)

print("\nCV ROC-AUC:", auc)

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission_catboost_cvV1.2.csv", index=False)

print("\nsubmission_catboost_cvV1.2.csv created")

## Catboost with Feature Engineering

In [ ]:

from sklearn.metrics import roc_auc_score

#Feature Engineering
train["avg_monthly_spend"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_monthly_spend"] = test["TotalCharges"] / (test["tenure"] + 1)

train["tenure_group"] = pd.cut(
    train["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

test["tenure_group"] = pd.cut(
    test["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

train["num_services"] = (train[service_cols] == "Yes").sum(axis=1)
test["num_services"] = (test[service_cols] == "Yes").sum(axis=1)

y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes(
    include=["object","string","category"]
).columns.tolist()


N_FOLDS = 5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== Fold {fold+1} =====")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(

        iterations=5000,
        learning_rate=0.03,
        depth=8,

        loss_function="Logloss",
        eval_metric="AUC",

        task_type="GPU",
        devices="0",

        random_seed=42,

        early_stopping_rounds=200,

        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features
    )

    val_pred = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(test)[:,1] / N_FOLDS


y_binary = y.map({"No":0,"Yes":1})

cv_auc = roc_auc_score(y_binary, oof_preds)

print("\nCV ROC-AUC:", cv_auc)


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission_catboost_cv_FEv1.csv", index=False)

print("\nsubmission_catboost_cv_fe.csv created")

## Catbost and Optuna Optimized

In [ ]:

import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
train["avg_monthly_spend"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_monthly_spend"] = test["TotalCharges"] / (test["tenure"] + 1)

train["tenure_group"] = pd.cut(
    train["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

test["tenure_group"] = pd.cut(
    test["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

train["num_services"] = (train[service_cols] == "Yes").sum(axis=1)
test["num_services"] = (test[service_cols] == "Yes").sum(axis=1)


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes(
    include=["object","string","category"]
).columns.tolist()


def objective(trial):

    params = {

        "iterations": 3000,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),

        "depth": trial.suggest_int("depth", 6, 10),

        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0, 1
        ),

        "random_strength": trial.suggest_float(
            "random_strength", 0, 1
        ),

        "loss_function": "Logloss",

        "eval_metric": "AUC",

        "task_type": "GPU",

        "devices": "0",

        "verbose": 0
    }

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    auc_scores = []

    for train_idx, val_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            cat_features=cat_features,
            early_stopping_rounds=100,
            verbose=False
        )

        preds = model.predict_proba(X_val)[:,1]

        y_val_binary = y_val.map({"No":0,"Yes":1})

        auc = roc_auc_score(y_val_binary, preds)

        auc_scores.append(auc)

    return np.mean(auc_scores)


study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)

print("\nBest AUC:", study.best_value)

print("\nBest parameters:")
print(study.best_params)

In [ ]:
best_params = study.best_params

final_model = CatBoostClassifier(

    iterations=5000,

    learning_rate=best_params["learning_rate"],
    depth=best_params["depth"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    bagging_temperature=best_params["bagging_temperature"],
    random_strength=best_params["random_strength"],

    loss_function="Logloss",
    eval_metric="AUC",

    task_type="GPU",
    devices="0",

    verbose=200
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

pred_probs = final_model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_optunaV1.csv", index=False)

print("\nSubmission file created: submission_catboost_optuna.csv")

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import subprocess
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

import optuna

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

N_FOLDS = 5
RANDOM_STATE = 42

DO_CATBOOST_OPTUNA = True        
CATBOOST_OPTUNA_TRIALS = 60     

CAT_ITERS = 5000

XGB_ROUNDS = 3000
LGB_ROUNDS = 3000

def has_nvidia_smi():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=3)
        return out.returncode == 0 and len(out.stdout.strip()) > 0
    except Exception:
        return False

try:
    import torch
    TORCH_CUDA = torch.cuda.is_available()
except Exception:
    TORCH_CUDA = False

def xgb_cuda_available():
    try:
        info = xgb.build_info()
        return info.get("USE_CUDA", False)
    except Exception:
        return False

def lgb_gpu_available():
    try:
        params = {"device_type": "gpu", "verbosity": -1}
        _ = lgb.Dataset(np.zeros((1,1)), label=[0])
        return True
    except Exception:
        return False

HAS_NVIDIA = has_nvidia_smi() or TORCH_CUDA
XGB_HAS_CUDA = xgb_cuda_available()
LGB_HAS_GPU = lgb_gpu_available()

print(f"GPU present (nvidia/torch): {HAS_NVIDIA}")
print(f"XGBoost GPU build: {XGB_HAS_CUDA}")
print(f"LightGBM GPU support (best-effort): {LGB_HAS_GPU}")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "Churn"
test_ids = test["id"].copy()

train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

train["avg_monthly_spend"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_monthly_spend"] = test["TotalCharges"] / (test["tenure"] + 1)

train["tenure_group"] = pd.cut(
    train["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)
test["tenure_group"] = pd.cut(
    test["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

service_cols = [
    "OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies"
]
train["num_services"] = (train[service_cols] == "Yes").sum(axis=1)
test["num_services"] = (test[service_cols] == "Yes").sum(axis=1)

y = train[TARGET].copy()
X = train.drop(columns=[TARGET]).copy()
X_test = test.copy()

cat_features = X.select_dtypes(include=["object","string","category"]).columns.tolist()
print("Detected categorical columns for CatBoost:", cat_features)

def compute_weights(scores):
    arr = np.array(scores)
    shifted = arr - 0.5
    shifted[shifted < 0] = 0
    if shifted.sum() == 0:
        shifted = arr - arr.min() + 1e-6
    weights = shifted / shifted.sum()
    return weights.tolist()

def optuna_tune_catboost(X, y, cat_features, n_trials=40):
    print("Running Optuna for CatBoost (fast CV inside objective)...")
    def objective(trial):
        params = {
            "iterations": 3000,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08),
            "depth": trial.suggest_int("depth", 6, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 8.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "task_type": "GPU" if HAS_NVIDIA else "CPU",
            "devices": "0" if HAS_NVIDIA else None,
            "verbose": 0,
        }
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            model = CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_features,
                      early_stopping_rounds=80, verbose=False)
            preds = model.predict_proba(X_val)[:,1]
            scores.append(roc_auc_score(y_val.map({"No":0,"Yes":1}), preds))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print("Optuna finished. Best AUC:", study.best_value)
    best = study.best_params
    final = {
        "iterations": CAT_ITERS,
        "learning_rate": best["learning_rate"],
        "depth": best["depth"],
        "l2_leaf_reg": best["l2_leaf_reg"],
        "bagging_temperature": best["bagging_temperature"],
        "random_strength": best["random_strength"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "task_type": "GPU" if HAS_NVIDIA else "CPU",
        "devices": "0" if HAS_NVIDIA else None,
        "verbose": 0
    }
    return final

def run_catboost_cv(X, y, X_test, cat_features, n_folds=5, params=None):
    print("Starting CatBoost CV...")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f" CatBoost fold {fold}/{n_folds}")
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**params)
        try:
            model.fit(X_tr, y_tr,
                      eval_set=(X_val, y_val),
                      cat_features=cat_features,
                      early_stopping_rounds=200,
                      verbose=200)
        except Exception as e:
            print(" CatBoost GPU error, retrying with CPU. Error:", e)
            params_cpu = params.copy()
            params_cpu["task_type"] = "CPU"
            params_cpu["devices"] = None
            model = CatBoostClassifier(**params_cpu)
            model.fit(X_tr, y_tr,
                      eval_set=(X_val, y_val),
                      cat_features=cat_features,
                      early_stopping_rounds=200,
                      verbose=200)

        val_pred = model.predict_proba(X_val)[:,1]
        oof[val_idx] = val_pred
        test_pred += model.predict_proba(X_test)[:,1] / n_folds
        fold_auc = roc_auc_score(y_val.map({"No":0,"Yes":1}), val_pred)
        print(f"  Fold {fold} AUC: {fold_auc:.6f}")
        scores.append(fold_auc)
        model.save_model(f"models/catboost_fold{fold}.cbm")
    mean_auc = np.mean(scores)
    print("CatBoost CV mean AUC:", mean_auc)
    return oof, test_pred, mean_auc

def run_xgb_cv(X, y, X_test, n_folds=5, params=None):
    print("Starting XGBoost CV...")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    X_all = pd.get_dummies(X, drop_first=False)
    X_test_dum = pd.get_dummies(X_test, drop_first=False)
    X_test_dum = X_test_dum.reindex(columns=X_all.columns, fill_value=0)

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []

    use_gpu = XGB_HAS_CUDA and HAS_NVIDIA
    params_train = params.copy()
    if use_gpu:
        params_train["tree_method"] = "hist"
    else:
        params_train["tree_method"] = "hist"
        params_train["device"] = "cpu"

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y), 1):
        print(f" XGBoost fold {fold}/{n_folds}")
        X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr.map({"No":0,"Yes":1}))
        dval = xgb.DMatrix(X_val, label=y_val.map({"No":0,"Yes":1}))
        dtest = xgb.DMatrix(X_test_dum)

        watchlist = [(dtrain, "train"), (dval, "valid")]

        try:
            booster = xgb.train(
                params_train,
                dtrain,
                num_boost_round=params_train.get("n_estimators", XGB_ROUNDS),
                evals=watchlist,
                early_stopping_rounds=150,
                verbose_eval=200
            )
        except Exception as e:
            print(" XGBoost GPU train error (or other). Falling back to CPU train. Error:", e)
            params_train_cpu = params.copy()
            params_train_cpu["device"] = "cpu"
            booster = xgb.train(
                params_train_cpu,
                dtrain,
                num_boost_round=params_train_cpu.get("n_estimators", XGB_ROUNDS),
                evals=watchlist,
                early_stopping_rounds=150,
                verbose_eval=200
            )

        val_pred = booster.predict(dval)
        oof[val_idx] = val_pred
        test_pred += booster.predict(dtest) / n_folds
        fold_auc = roc_auc_score(y_val.map({"No":0,"Yes":1}), val_pred)
        print(f"  Fold {fold} AUC: {fold_auc:.6f}")
        scores.append(fold_auc)
        booster.save_model(f"models/xgb_fold{fold}.json")
    mean_auc = np.mean(scores)
    print("XGBoost CV mean AUC:", mean_auc)
    return oof, test_pred, mean_auc

def run_lgb_cv(X, y, X_test, n_folds=5, params=None):
    print("Starting LightGBM CV...")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    X_all = pd.get_dummies(X, drop_first=False)
    X_test_dum = pd.get_dummies(X_test, drop_first=False)
    X_test_dum = X_test_dum.reindex(columns=X_all.columns, fill_value=0)

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []

    use_gpu = LGB_HAS_GPU and HAS_NVIDIA
    params_train = params.copy()
    if use_gpu:
        params_train["device_type"] = "gpu"
        params_train["gpu_platform_id"] = 0
        params_train["gpu_device_id"] = 0
    else:
        params_train["device_type"] = "cpu"

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y), 1):
        print(f" LightGBM fold {fold}/{n_folds}")
        X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr.map({"No":0,"Yes":1}))
        dval = lgb.Dataset(X_val, label=y_val.map({"No":0,"Yes":1}))

        try:
            bst = lgb.train(
                params_train,
                dtrain,
                num_boost_round=params_train.get("n_estimators", LGB_ROUNDS),
                valid_sets=[dtrain, dval],
                early_stopping_rounds=150,
                verbose_eval=200
            )
        except Exception as e:
            print(" LightGBM GPU error or other, falling back to CPU. Error:", e)
            params_cpu = params.copy()
            if "device_type" in params_cpu:
                params_cpu.pop("device_type", None)
            bst = lgb.train(
                params_cpu,
                dtrain,
                num_boost_round=params_cpu.get("n_estimators", LGB_ROUNDS),
                valid_sets=[dtrain, dval],
                early_stopping_rounds=150,
                verbose_eval=200
            )

        val_pred = bst.predict(X_val)
        oof[val_idx] = val_pred
        test_pred += bst.predict(X_test_dum) / n_folds
        fold_auc = roc_auc_score(y_val.map({"No":0,"Yes":1}), val_pred)
        print(f"  Fold {fold} AUC: {fold_auc:.6f}")
        scores.append(fold_auc)
        bst.save_model(f"models/lgb_fold{fold}.txt")
    mean_auc = np.mean(scores)
    print("LightGBM CV mean AUC:", mean_auc)
    return oof, test_pred, mean_auc


catboost_params_default = {
    "iterations": CAT_ITERS,
    "learning_rate": 0.03,
    "depth": 8,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "task_type": "GPU" if HAS_NVIDIA else "CPU",
    "devices": "0" if HAS_NVIDIA else None,
    "verbose": 0
}

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": 0.03,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "lambda": 1.0,
    "alpha": 0.0,
    "n_estimators": XGB_ROUNDS
}

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.03,
    "num_leaves": 127,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "n_estimators": LGB_ROUNDS
}

if DO_CATBOOST_OPTUNA:
    catboost_params = optuna_tune_catboost(X, y, cat_features, n_trials=CATBOOST_OPTUNA_TRIALS)
else:
    catboost_params = catboost_params_default

catboost_params["verbose"] = 0

cat_oof, cat_test, cat_auc = run_catboost_cv(X, y, X_test, cat_features, n_folds=N_FOLDS, params=catboost_params)

xgb_oof, xgb_test, xgb_auc = run_xgb_cv(X, y, X_test, n_folds=N_FOLDS, params=xgb_params)

lgb_oof, lgb_test, lgb_auc = run_lgb_cv(X, y, X_test, n_folds=N_FOLDS, params=lgb_params)

print("\nPer-model CV AUCs:")
print(f"  CatBoost: {cat_auc:.6f}")
print(f"  XGBoost : {xgb_auc:.6f}")
print(f"  LightGBM: {lgb_auc:.6f}")

scores = [cat_auc, xgb_auc, lgb_auc]
weights = compute_weights(scores)
print("\nEnsemble weights (CatBoost, XGB, LGB):", weights)

ensemble_test = weights[0]*cat_test + weights[1]*xgb_test + weights[2]*lgb_test
oof_ensemble = weights[0]*cat_oof + weights[1]*xgb_oof + weights[2]*lgb_oof
oof_auc = roc_auc_score(y.map({"No":0,"Yes":1}), oof_ensemble)
print("OOF Ensemble AUC:", oof_auc)

submission = pd.DataFrame({"id": test_ids, "Churn": ensemble_test})
submission.to_csv("submission_ensemble.csv", index=False)
print("\nSaved submission_ensemble.csv")

GPU present (nvidia/torch): True
XGBoost GPU build: True
LightGBM GPU support (best-effort): True


[I 2026-03-11 22:23:55,984] A new study created in memory with name: no-name-82963f6a-235c-4e91-8a9e-1be571dce462


Detected categorical columns for CatBoost: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']
Running Optuna for CatBoost (fast CV inside objective)...


  0%|          | 0/60 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


...

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8895490	best: 0.8895490 (0)	total: 94ms	remaining: 7m 49s
200:	test: 0.9149982	best: 0.9149982 (200)	total: 18.7s	remaining: 7m 25s
400:	test: 0.9160586	best: 0.9160586 (400)	total: 37.1s	remaining: 7m 5s
600:	test: 0.9165893	best: 0.9165893 (600)	total: 55.8s	remaining: 6m 48s
800:	test: 0.9168705	best: 0.9168705 (800)	total: 1m 14s	remaining: 6m 29s
1000:	test: 0.9170761	best: 0.9170761 (1000)	total: 1m 33s	remaining: 6m 11s
1200:	test: 0.9172166	best: 0.9172166 (1200)	total: 1m 51s	remaining: 5m 53s
1400:	test: 0.9173212	best: 0.9173219 (1397)	total: 2m 10s	remaining: 5m 34s
1600:	test: 0.9173865	best: 0.9173865 (1600)	total: 2m 29s	remaining: 5m 16s
1800:	test: 0.9174602	best: 0.9174614 (1798)	total: 2m 47s	remaining: 4m 58s
2000:	test: 0.9175090	best: 0.9175092 (1981)	total: 3m 6s	remaining: 4m 39s
2200:	test: 0.9175397	best: 0.9175435 (2182)	total: 3m 25s	remaining: 4m 21s
2400:	test: 0.9175555	best: 0.9175569 (2396)	total: 3m 44s	remaining: 4m 2s
2600:	test: 0.9175569

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8882197	best: 0.8882197 (0)	total: 95.7ms	remaining: 7m 58s
200:	test: 0.9124334	best: 0.9124334 (200)	total: 18.7s	remaining: 7m 27s
400:	test: 0.9134335	best: 0.9134335 (400)	total: 37.3s	remaining: 7m 7s
600:	test: 0.9139538	best: 0.9139538 (600)	total: 55.9s	remaining: 6m 49s
800:	test: 0.9142366	best: 0.9142366 (800)	total: 1m 14s	remaining: 6m 29s
1000:	test: 0.9143932	best: 0.9143932 (1000)	total: 1m 32s	remaining: 6m 11s
1200:	test: 0.9144838	best: 0.9144838 (1200)	total: 1m 51s	remaining: 5m 52s
1400:	test: 0.9145833	best: 0.9145833 (1400)	total: 2m 10s	remaining: 5m 34s
1600:	test: 0.9146201	best: 0.9146204 (1583)	total: 2m 28s	remaining: 5m 16s
1800:	test: 0.9146643	best: 0.9146643 (1799)	total: 2m 47s	remaining: 4m 57s
2000:	test: 0.9146890	best: 0.9146911 (1995)	total: 3m 6s	remaining: 4m 39s
2200:	test: 0.9147103	best: 0.9147108 (2197)	total: 3m 25s	remaining: 4m 21s
2400:	test: 0.9147179	best: 0.9147244 (2291)	total: 3m 44s	remaining: 4m 2s
2600:	test: 0.91472

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata


def run_lgb_cv(X, y, X_test, n_folds=5):

    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=42
    )

    X_all = pd.get_dummies(X)
    X_test_dum = pd.get_dummies(X_test)

    X_test_dum = X_test_dum.reindex(
        columns=X_all.columns,
        fill_value=0
    )

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    scores = []

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.03,
        "num_leaves": 127,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "lambda_l2": 1.0,
        "verbosity": -1
    }

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all, y)):

        print(f"\nLightGBM Fold {fold+1}")

        X_tr = X_all.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]

        X_val = X_all.iloc[val_idx]
        y_val = y.iloc[val_idx]

        dtrain = lgb.Dataset(
            X_tr,
            label=y_tr.map({"No":0,"Yes":1})
        )

        dval = lgb.Dataset(
            X_val,
            label=y_val.map({"No":0,"Yes":1})
        )

        bst = lgb.train(
            params,
            dtrain,
            num_boost_round=3000,
            valid_sets=[dtrain, dval],
            callbacks=[
                lgb.early_stopping(150),
                lgb.log_evaluation(200)
            ]
        )

        val_pred = bst.predict(X_val)

        oof[val_idx] = val_pred

        test_pred += bst.predict(X_test_dum) / n_folds

        fold_auc = roc_auc_score(
            y_val.map({"No":0,"Yes":1}),
            val_pred
        )

        print("Fold AUC:", fold_auc)

        scores.append(fold_auc)

    mean_auc = np.mean(scores)

    print("\nLightGBM CV AUC:", mean_auc)

    return oof, test_pred, mean_auc


lgb_oof, lgb_test, lgb_auc = run_lgb_cv(
    X,
    y,
    X_test,
    n_folds=5
)


scores = np.array([
    cat_auc,
    xgb_auc,
    lgb_auc
])

weights = (scores - 0.5)
weights = weights / weights.sum()

print("\nModel AUCs:", scores)
print("Ensemble weights:", weights)


weighted_test = (
    weights[0] * cat_test +
    weights[1] * xgb_test +
    weights[2] * lgb_test
)



cat_rank = rankdata(cat_test)
xgb_rank = rankdata(xgb_test)
lgb_rank = rankdata(lgb_test)

rank_avg = (
    weights[0] * cat_rank +
    weights[1] * xgb_rank +
    weights[2] * lgb_rank
)

rank_avg = rank_avg / rank_avg.max()



final_pred = 0.5 * weighted_test + 0.5 * rank_avg



weighted_oof = (
    weights[0] * cat_oof +
    weights[1] * xgb_oof +
    weights[2] * lgb_oof
)

ensemble_auc = roc_auc_score(
    y.map({"No":0,"Yes":1}),
    weighted_oof
)

print("\nFinal Ensemble OOF AUC:", ensemble_auc)


submission = pd.DataFrame({
    "id": test_ids,
    "Churn": final_pred
})

submission.to_csv(
    "submission_ensemble_final.csv",
    index=False
)

print("\nSubmission saved → submission_ensemble_final.csv")


LightGBM Fold 1
Training until validation scores don't improve for 150 rounds
[200]	training's auc: 0.920303	valid_1's auc: 0.915452
[400]	training's auc: 0.924573	valid_1's auc: 0.915968
[600]	training's auc: 0.927905	valid_1's auc: 0.915978
Early stopping, best iteration is:
[470]	training's auc: 0.925777	valid_1's auc: 0.915989
Fold AUC: 0.9159893080158393

LightGBM Fold 2
Training until validation scores don't improve for 150 rounds
[200]	training's auc: 0.920026	valid_1's auc: 0.916605
[400]	training's auc: 0.924325	valid_1's auc: 0.917046
[600]	training's auc: 0.927643	valid_1's auc: 0.917125
[800]	training's auc: 0.930628	valid_1's auc: 0.9171
Early stopping, best iteration is:
[686]	training's auc: 0.928958	valid_1's auc: 0.917144
Fold AUC: 0.9171443331205821

LightGBM Fold 3
Training until validation scores don't improve for 150 rounds
[200]	training's auc: 0.920217	valid_1's auc: 0.915877
[400]	training's auc: 0.924453	valid_1's auc: 0.916321
[600]	training's auc: 0.927792	v